# 07 - Leakage-safe prospective dataset

## Purpose
This notebook creates the **first leakage-safe modelling dataset** for Paper 2. It does not train a model. It fixes the prediction target, protects the time order, and records the decisions needed before modelling.

### Prediction contract (version 1)
At a non-injury observation at time $t$, predict whether an **incident injury episode starts within the next 28 Date units**. Only features recorded at time $t$ are retained. `Date`, `Athlete ID`, the current `injury` label, and every engineered temporal-gap variable are excluded from features.

The 28-unit horizon is a provisional, data-informed choice: the audit shows that a 7- or 14-unit horizon has no positive labels, while recorded incident injuries begin at least 22 units after the preceding record. Confirm the clinical meaning of `Date` and this horizon with the supervisor/data documentation before treating it as final.

A row is eligible only when the athlete has observation coverage through the end of the horizon, unless an incident injury is observed before then. A 28-unit post-event washout removes rows immediately after a recorded injury episode.

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data/raw/data_weekly.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

RAW_PATH = PROJECT_ROOT / 'data/raw/data_weekly.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = 'Athlete ID'
DATE_COL = 'Date'
CURRENT_LABEL = 'injury'
HORIZON_DAYS = 28       # Provisional: see prediction contract above.
WASHOUT_DAYS = 28       # Rows this close to a prior incident episode are excluded.
RANDOM_STATE = 42


## Step 1 - Load and audit temporal structure

`Date` is used only to construct and audit the future label. It is never a model feature.

In [25]:
df = pd.read_csv(RAW_PATH)
required = {ID_COL, DATE_COL, CURRENT_LABEL}
missing = required - set(df.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

df = df.sort_values([ID_COL, DATE_COL], kind='stable').reset_index(drop=True)
assert not df.duplicated([ID_COL, DATE_COL]).any(), 'Duplicate athlete-date pairs must be resolved first.'
assert set(df[CURRENT_LABEL].dropna().unique()).issubset({0, 1}), 'injury must be binary.'

df['_previous_date'] = df.groupby(ID_COL)[DATE_COL].shift(1)
df['_previous_injury'] = df.groupby(ID_COL)[CURRENT_LABEL].shift(1).fillna(0).astype(int)
df['_previous_gap'] = df[DATE_COL] - df['_previous_date']

print(f'Rows: {len(df):,}; athletes: {df[ID_COL].nunique():,}; current injury rows: {int(df[CURRENT_LABEL].sum()):,}')
display(df[[ID_COL, DATE_COL, CURRENT_LABEL, '_previous_date', '_previous_gap']].head())


Rows: 42,798; athletes: 74; current injury rows: 575


,Athlete ID,Date,injury,_previous_date,_previous_gap
0,0,0,0,NaN,NaN
1,0,1,0,0.0,1.0
2,0,2,0,1.0,1.0
3,0,3,0,2.0,1.0
4,0,4,0,3.0,1.0


## Step 2 - Identify incident injury episodes

Several consecutive `injury = 1` rows are treated as one episode. Only its first row is an incident start. This prevents the model from counting the same injury repeatedly.

In [26]:
df['_incident_start'] = (df[CURRENT_LABEL].eq(1) & df['_previous_injury'].eq(0))
incident_rows = df.loc[df['_incident_start'], [ID_COL, DATE_COL, '_previous_gap']].copy()
incident_rows = incident_rows.rename(columns={'_previous_gap': 'gap_before_incident'})

print(f'Incident injury episodes: {len(incident_rows):,}')
display(incident_rows['gap_before_incident'].describe(percentiles=[.10, .25, .50, .75, .90, .95]))

for horizon in (7, 14, 28):
    count = int((incident_rows['gap_before_incident'] <= horizon).sum())
    print(f'Incident injuries preceded by a gap <= {horizon}: {count:,}')


Incident injury episodes: 389


count    380.000000
mean      26.902632
std       22.281327
min       22.000000
10%       22.000000
25%       22.000000
50%       22.000000
75%       22.000000
90%       22.000000
95%       58.050000
max      259.000000
Name: gap_before_incident, dtype: float64

Incident injuries preceded by a gap <= 7: 0
Incident injuries preceded by a gap <= 14: 0
Incident injuries preceded by a gap <= 28: 354


## Step 3 - Create the future outcome without using a gap feature

For each athlete-week, find the next incident event after $t$. The target is 1 if that event begins within the selected horizon. This use of dates is label construction, not feature engineering. The model will never receive the elapsed time to the next record.

In [27]:
def build_future_target(group: pd.DataFrame) -> pd.DataFrame:
    group = group.copy()
    dates = group[DATE_COL].to_numpy()
    event_dates = group.loc[group['_incident_start'], DATE_COL].to_numpy()

    next_event_position = np.searchsorted(event_dates, dates, side='right')
    next_event_date = np.full(len(group), np.nan)
    has_next_event = next_event_position < len(event_dates)
    next_event_date[has_next_event] = event_dates[next_event_position[has_next_event]]

    group['_next_incident_date'] = next_event_date
    group['_future_incident_in_horizon'] = (
        pd.notna(group['_next_incident_date'])
        & (group['_next_incident_date'] <= group[DATE_COL] + HORIZON_DAYS)
    ).astype(int)

    # A negative requires observation coverage to the horizon. A positive is known when the event occurs.
    group['_has_horizon_coverage'] = group[DATE_COL] + HORIZON_DAYS <= dates.max()

    # Exclude the injury rows themselves and the recovery/washout period after any earlier incident start.
    prior_event_position = np.searchsorted(event_dates, dates, side='right') - 1
    most_recent_event = np.full(len(group), np.nan)
    has_prior_event = prior_event_position >= 0
    most_recent_event[has_prior_event] = event_dates[prior_event_position[has_prior_event]]
    days_since_event = dates - most_recent_event
    in_washout = pd.notna(most_recent_event) & (days_since_event <= WASHOUT_DAYS)

    group['_eligible_index'] = (
        group[CURRENT_LABEL].eq(0)
        & ~in_washout
        & (group['_has_horizon_coverage'] | group['_future_incident_in_horizon'].eq(1))
    )
    return group

labelled = (df.groupby(ID_COL, group_keys=False, sort=False)
              .apply(build_future_target, include_groups=False)
              .reset_index())

# groupby.apply removes the group key from the frame in current pandas; restore it safely if needed.
if ID_COL not in labelled.columns:
    labelled[ID_COL] = df[ID_COL].to_numpy()

labelled['_future_incident_in_horizon'] = labelled['_future_incident_in_horizon'].astype(int)
print('Eligible rows:', int(labelled['_eligible_index'].sum()))
print('Positive future incident labels:', int(labelled.loc[labelled['_eligible_index'], '_future_incident_in_horizon'].sum()))
display(labelled.loc[labelled['_eligible_index'], '_future_incident_in_horizon'].value_counts().rename('n'))


Eligible rows: 40180
Positive future incident labels: 2245


_future_incident_in_horizon
0    37935
1     2245
Name: n, dtype: int64

## Step 4 - Freeze the allowed feature set

The feature set is restricted to the original 69 workload/recovery variables. No `gap`, no future outcome, no athlete identifier, and no date can enter the model. Imputation, scaling, clipping, balancing, calibration, and tuning belong later inside training folds.

In [28]:
METADATA_COLUMNS = {
    ID_COL, DATE_COL, CURRENT_LABEL, '_previous_date', '_previous_injury', '_previous_gap',
    '_incident_start', '_next_incident_date', '_future_incident_in_horizon',
    '_has_horizon_coverage', '_eligible_index'
}
feature_cols = [c for c in df.columns if c not in METADATA_COLUMNS]
forbidden = {'gap', 'time_gap', ID_COL, DATE_COL, CURRENT_LABEL}
assert not any(c.lower() in forbidden for c in feature_cols), 'A forbidden feature entered the model matrix.'
assert len(feature_cols) == 69, f'Expected 69 original features; found {len(feature_cols)}.'
assert all(pd.api.types.is_numeric_dtype(labelled[c]) for c in feature_cols), 'Features must be numeric.'

model_df = labelled.loc[labelled['_eligible_index'], [ID_COL, DATE_COL, *feature_cols, '_future_incident_in_horizon']].copy()
model_df = model_df.rename(columns={'_future_incident_in_horizon': 'future_incident_28d'})

assert model_df[feature_cols].columns.tolist() == feature_cols
assert not model_df.columns.duplicated().any()
display(model_df.head())
print(f'Model rows: {len(model_df):,}; features: {len(feature_cols)}; target prevalence: {model_df.future_incident_28d.mean():.4%}')


,Athlete ID,Date,nr. sessions,nr. rest days,total kms,max km one day,total km Z3-Z4-Z5-T1-T2,"nr. tough sessions (effort in Z5, T1 or T2)",nr. days with interval session,total km Z3-4,...,avg training success.2,min training success.2,max training success.2,avg recovery.2,min recovery.2,max recovery.2,rel total kms week 0_1,rel total kms week 0_2,rel total kms week 1_2,future_incident_28d
0,0,0,5.0,2.0,22.2,16.4,11.8,1.0,2.0,10.0,...,0.0,0.0,0.0,0.18,0.16,0.20,0.718447,1.378882,1.919255,0
1,0,1,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,...,0.0,0.0,0.0,0.18,0.16,0.20,0.683544,1.018868,1.490566,0
2,0,2,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,...,0.0,0.0,0.0,0.17,0.16,0.18,0.683544,1.018868,1.490566,0
3,0,3,5.0,2.0,21.6,16.4,11.7,1.0,2.0,10.0,...,0.0,0.0,0.0,0.18,0.16,0.18,0.683544,1.018868,1.490566,0
4,0,4,6.0,1.0,39.2,17.6,18.9,1.0,3.0,17.2,...,0.0,0.0,0.0,0.17,0.16,0.18,2.202247,1.361111,0.618056,0


Model rows: 40,180; features: 69; target prevalence: 5.5874%


## Step 5 - Save dataset and reproducibility audit

The next notebook must load this saved dataset and split by athlete before fitting any preprocessing or model.

In [29]:
dataset_path = PROCESSED_DIR / f'leakage_safe_prospective_{HORIZON_DAYS}d.csv'
audit_path = PROCESSED_DIR / f'leakage_safe_prospective_{HORIZON_DAYS}d_audit.csv'

audit = pd.DataFrame([{
    'horizon_date_units': HORIZON_DAYS,
    'washout_date_units': WASHOUT_DAYS,
    'source_rows': len(df),
    'athletes': df[ID_COL].nunique(),
    'current_injury_rows': int(df[CURRENT_LABEL].sum()),
    'incident_injury_episodes': int(df['_incident_start'].sum()),
    'eligible_index_rows': len(model_df),
    'future_incident_positives': int(model_df['future_incident_28d'].sum()),
    'future_incident_prevalence': model_df['future_incident_28d'].mean(),
    'n_features': len(feature_cols),
    'forbidden_features': 'Athlete ID; Date; injury; gap/time_gap; all future-label metadata'
}])

model_df.to_csv(dataset_path, index=False)
audit.to_csv(audit_path, index=False)
display(audit.T)
print(f'Saved: {dataset_path.relative_to(PROJECT_ROOT)}')
print(f'Saved: {audit_path.relative_to(PROJECT_ROOT)}')


,0
horizon_date_units,28
washout_date_units,28
source_rows,42798
athletes,74
current_injury_rows,575
incident_injury_episodes,389
eligible_index_rows,40180
future_incident_positives,2245
future_incident_prevalence,0.055874
n_features,69


Saved: data/processed/leakage_safe_prospective_28d.csv
Saved: data/processed/leakage_safe_prospective_28d_audit.csv


## Required decision before Notebook 08

Before training, verify with the dataset documentation/supervisor that one `Date` unit is a day and that a 28-day forecast matches the clinical question. If it does not, update `HORIZON_DAYS`, rerun this notebook, and record the reason in the prediction contract. Do not proceed with the original concurrent `injury` label or the `gap` feature as a prospective prediction claim.